# CMU 10-799 HW1 Part II Runner

Use this notebook to run the Part II DDPM training and sampling workflow on Google Colab.

The source of truth for code is the repository, not this notebook. Edit code locally, commit/push, then pull the latest code here before training.


## 0. GPU Runtime

In Colab, use `Runtime -> Change runtime type -> GPU`, then run these checks.


In [ ]:
!nvidia-smi

import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")


## 1. Mount Drive

Colab local disk disappears when the runtime resets. Save long training outputs to Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/cmu-10799-diffusion"
!mkdir -p "{DRIVE_ROOT}/logs" "{DRIVE_ROOT}/checkpoints" "{DRIVE_ROOT}/samples"
print(DRIVE_ROOT)


## 2. Clone Or Update Repo

If this is a fresh runtime, clone your fork. If it already exists, pull the latest code.


In [ ]:
%cd /content

import os

repo_dir = "/content/cmu-10799-diffusion"
repo_url = "https://github.com/Eryc123Y/cmu-10799-diffusion.git"  # Change if needed.

if os.path.isdir(repo_dir):
    %cd /content/cmu-10799-diffusion
    !git pull origin main
else:
    !git clone "{repo_url}"
    %cd /content/cmu-10799-diffusion

!git status --short --branch
!git log -1 --oneline


## 3. Install Dependencies

Colab usually already has CUDA PyTorch. This cell installs the extra packages used by the starter code.


In [ ]:
!pip install -q einops PyYAML tqdm scipy pandas matplotlib datasets huggingface-hub torch-fidelity wandb

import yaml
import einops
import datasets
import torch_fidelity

print("imports ok")


## 4. Create A Colab Config

This is a practical starter config. If memory is tight, reduce `batch_size` first.


In [ ]:
%%writefile configs/ddpm_colab.yaml
data:
  dataset: "celeba"
  root: "./data/celeba-subset"
  from_hub: true
  repo_name: "electronickale/cmu-10799-celeba64-subset"
  image_size: 64
  channels: 3
  num_workers: 2
  pin_memory: true
  augment: true

model:
  base_channels: 64
  channel_mult: [1, 2, 2, 4]
  num_res_blocks: 2
  attention_resolutions: [16]
  num_heads: 4
  dropout: 0.1
  use_scale_shift_norm: true

training:
  batch_size: 64
  learning_rate: 0.0002
  weight_decay: 0.0
  betas: [0.9, 0.999]
  ema_decay: 0.9999
  ema_start: 1000
  gradient_clip_norm: 1.0
  num_iterations: 100000
  log_every: 100
  sample_every: 2000
  save_every: 5000
  num_samples: 16

ddpm:
  num_timesteps: 1000
  beta_start: 0.0001
  beta_end: 0.02

sampling:
  num_steps: 1000
  sampler: "ddpm"

infrastructure:
  seed: 42
  device: "cuda"
  num_gpus: 1
  mixed_precision: true
  compile_model: false

checkpoint:
  dir: "./checkpoints"
  resume: null

logging:
  dir: "/content/drive/MyDrive/cmu-10799-diffusion/logs"
  wandb:
    enabled: false
    project: "cmu-10799-diffusion"
    entity: null


## 5. Syntax And Tiny Tensor Smoke Test

Run this before any long training. It checks importability, model output shape, DDPM loss, and a tiny sampling loop.


In [ ]:
!python -m compileall -q src train.py sample.py

import torch
from src.models.unet import UNet
from src.methods.ddpm import DDPM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    in_channels=3,
    out_channels=3,
    base_channels=64,
    channel_mult=(1, 2, 2, 4),
    num_res_blocks=1,
    attention_resolutions=[16],
    num_heads=4,
    dropout=0.1,
    use_scale_shift_norm=True,
    image_size=64,
).to(device)

method = DDPM(
    model=model,
    device=device,
    num_timesteps=10,
    beta_start=1e-4,
    beta_end=2e-2,
)

x_0 = torch.randn(2, 3, 64, 64, device=device)
t = torch.randint(0, method.num_timesteps, (2,), device=device)

x_t, noise = method.forward_process(x_0, t)
loss, metrics = method.compute_loss(x_0)
samples = method.sample(batch_size=2, image_shape=(3, 64, 64), num_steps=10)

print("x_t:", x_t.shape)
print("noise:", noise.shape)
print("loss:", loss.shape, metrics)
print("samples:", samples.shape)


## 6. Optional: Download Dataset To Local Disk

The config uses `from_hub: true`, so this may not be necessary. If loading from hub is slow, download/cache the dataset locally.


In [ ]:
# Optional. Uncomment if you want a local cache.
# !python download_dataset.py --output_dir ./data/celeba-subset --split all


## 7. Single-Batch Sanity Check

Run this before full training. If this fails to lower loss or crashes during sampling, full training will waste GPU time.


In [ ]:
# This should be much cheaper than full training.
# Stop manually once the behavior is clear.
!python train.py --method ddpm --config configs/ddpm_colab.yaml --overfit-single-batch


## 8. Full Training

Record the GPU type, start/end time, final iteration, loss curve, sample grids, and checkpoint path for the report.


In [ ]:
!python train.py --method ddpm --config configs/ddpm_colab.yaml


## 9. Generate A 16-Sample Grid

Update `CHECKPOINT_PATH` to the checkpoint you want to evaluate.


In [ ]:
CHECKPOINT_PATH = "TODO_REPLACE_WITH_CHECKPOINT.pt"

!python sample.py \
  --checkpoint "{CHECKPOINT_PATH}" \
  --method ddpm \
  --num_samples 16 \
  --batch_size 16 \
  --grid \
  --output /content/drive/MyDrive/cmu-10799-diffusion/samples/part_ii_16_grid.png


## 10. Evidence To Copy Into The Report

- Model size: printed by `train.py` at startup.
- Batch size: `training.batch_size` in `configs/ddpm_colab.yaml`.
- Total training iterations: `training.num_iterations` and final checkpoint step.
- Training loss curve: console logs, saved logs, or wandb if enabled.
- Compute cost: GPU type from `nvidia-smi` times wall-clock hours.
- 16-sample grid: `/content/drive/MyDrive/cmu-10799-diffusion/samples/part_ii_16_grid.png`.
- KID: generate 1k samples and evaluate with a KID script after the baseline model is trained.
